In [ ]:
use_gpu = True

if use_gpu:
    import tensorly as tl
    tl.set_backend('pytorch')

In [ ]:
import cell2cell as c2c
import liana as li

import pandas as pd
#import decoupler as dc
import scanpy as sc

import plotnine as p9
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
import logging
logging.disable(logging.WARNING) # This is to avoid too many logs because of GSEApy

import warnings
warnings.filterwarnings('ignore')

In [ ]:
#working env
import os
os.chdir(r'\\isdsyn1-1.isd.med.uni-muenchen.de\BD-Bernhagen\Simon Ebert\snRNAseq_Python_env_normalMT')
os.listdir()

In [ ]:
%pip freeze > requirements.txt

In [ ]:
data_folder = '//isdsyn1-1.isd.med.uni-muenchen.de/BD-Bernhagen/Simon Ebert/snRNAseq_Python_env_normalMT/cell2cell_tensor/output/quickstart/'
output_folder = '//isdsyn1-1.isd.med.uni-muenchen.de/BD-Bernhagen/Simon Ebert/snRNAseq_Python_env_normalMT/cell2cell_tensor/output/'
c2c.io.directories.create_directory(data_folder)
c2c.io.directories.create_directory(output_folder)

In [ ]:
adata = c2c.datasets.balf_covid("integrated_harmony.h5ad")

In [ ]:

adata=adata[adata.obs.cell_type != "ECs2"]
adata.obs.cell_type.value_counts()

In [ ]:
adata.layers

In [ ]:
# save the raw counts to a layer
adata.X=adata.layers["soupX_counts"].copy()
adata.layers["counts"] = adata.X.copy()

# Normalize the data
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
adata.obs.head()

In [ ]:
li.method.show_methods()

In [ ]:
liana_resources = li.resource.show_resources()
print(*liana_resources, sep = ', ')

In [ ]:
lr_pairs = li.resource.select_resource('mouseconsensus')

In [ ]:
li.mt.rank_aggregate.by_sample(adata, 
                               sample_key='batch', 
                               groupby='cell_type', 
                               resource_name = 'mouseconsensus',
                               expr_prop=0.1, # must be expressed in expr_prop fraction of cells
                               min_cells = 5,
                               n_perms = 100, 
                               use_raw = False, # run on log- and library-normalized counts
                               verbose = True, 
                               inplace = True
                              )

In [ ]:
adata.uns['liana_res'].head()

In [ ]:
plot = li.pl.dotplot_by_sample(adata=adata,
                               colour='magnitude_rank',
                               size='specificity_rank',
                               source_labels=["Ms"],
                               target_labels=["Ms"],
                               ligand_complex=['Fn1'],
                               receptor_complex=['Itga9'],
                               sample_key='batch',
                               inverse_colour=True,
                               inverse_size=True,
                               figure_size=(14, 10),
                               size_range=(1, 6),
                               )

plot.save(output_folder + '/Dotplot-by-sample.pdf', height=9, width=9)
plot

In [ ]:
adata.uns['liana_res'].to_csv(output_folder + '/LIANA_by_sample.csv', index=False)

In [ ]:
adata.write_h5ad(output_folder + '/adata_processed.h5ad', compression='gzip')

In [ ]:
sorted_samples = sorted(adata.obs['batch'].unique())


In [ ]:
sorted_samples

In [ ]:
# build the tensor
tensor = li.multi.to_tensor_c2c(liana_res=adata.uns['liana_res'], # LIANA's dataframe containing results
                                sample_key='batch', # Column name of the samples
                                source_key='source', # Column name of the sender cells
                                target_key='target', # Column name of the receiver cells
                                ligand_key='ligand_complex', # Column name of the ligands
                                receptor_key='receptor_complex', # Column name of the receptors
                                score_key='magnitude_rank', # Column name of the communication scores to use
                                inverse_fun=lambda x: 1 - x, # Transformation function
                                how='outer', # What to include across all samples
                                outer_fraction=1/3., # Fraction of samples as threshold to include cells and LR pairs.
                                context_order=sorted_samples, # Order to store the contexts in the tensor
                               )

In [ ]:
tensor.shape

In [ ]:
tensor.missing_fraction()

In [ ]:
tensor.sparsity_fraction()


In [ ]:
tensor.excluded_value_fraction()

In [ ]:
from collections import defaultdict

element_dict = defaultdict(lambda: 'Unknown')
context_dict = element_dict.copy()


In [ ]:
context_dict.update({'HFD' : 'HFD',
                     'HFD_L1' : 'HFD_L1',

                    })
dimensions_dict = [context_dict, None, None, None]

In [ ]:
dimensions_dict = [context_dict, None, None, None]
meta_tensor = c2c.tensor.generate_tensor_metadata(interaction_tensor=tensor,
                                                  metadata_dicts=dimensions_dict,
                                                  fill_with_order_elements=True
                                                 )

In [ ]:
c2c.io.export_variable_with_pickle(variable=tensor,
                                   filename=output_folder + '/Tensor.pkl')
c2c.io.export_variable_with_pickle(variable=meta_tensor,
                                   filename=output_folder + '/Tensor-Metadata.pkl')

In [ ]:
tensor = c2c.io.read_data.load_tensor(output_folder + '/Tensor.pkl')
meta_tensor = c2c.io.load_variable_with_pickle(output_folder + '/Tensor-Metadata.pkl')

In [ ]:
tensor

In [ ]:
meta_tensor

In [ ]:
c2c.analysis.run_tensor_cell2cell_pipeline(tensor,
                                           meta_tensor,
                                           rank=None, # Number of factors to perform the factorization. If None, it is automatically determined by an elbow analysis
                                           tf_optimization='robust', # To define how robust we want the analysis to be.
                                           random_state=0, # Random seed for reproducibility
                                           device='cuda', # Device to use. If using GPU and PyTorch, use 'cuda'. For CPU use 'cpu'
                                           output_folder=output_folder, # Whether to save the figures in files. If so, a folder pathname must be passed
                                           upper_rank=25, # Maximum number of factors to test
                                           
                                          )

In [ ]:
tensor.factors.keys()

In [ ]:
tensor.factors['Contexts']

In [ ]:
tensor.factors['Contexts']

In [ ]:
context_dict

In [ ]:
metadictfactors = c2c.io.load_tensor_factors(output_folder + '/Loadings.xlsx')

In [ ]:
from collections import defaultdict

element_dict = defaultdict(lambda: 'Unknown')

In [ ]:
context_dict = element_dict.copy()

context_dict.update({'HFD' : 'HFD',
                     'HFD_L1' : 'HFD_L1',
                     
                    })

In [ ]:
import cell2cell as c2c

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import importlib
importlib.reload(sns)

In [ ]:
sns.__version__

In [ ]:
%pip install statannotations==0.5.0

In [ ]:
groups_order = ['HFD', "HFD_L1"]
fig_filename = output_folder + '/BALF-Severity-Boxplots.pdf'

_ = c2c.plotting.context_boxplot(context_loadings=tensor.factors['Contexts'],
                                 metadict=context_dict,
                                 nrows=2,
                                 figsize=(16, 12),
                                 group_order=groups_order,
                                 statistical_test='t-test_ind',
                                 pval_correction='fdr_bh',
                                 cmap='plasma',
                                 verbose=False, 
                                 filename=fig_filename,
                                )

In [ ]:
fig_filename = output_folder + '/Clustermap-LRs.pdf'

_ = c2c.plotting.loading_clustermap(loadings=tensor.factors['Ligand-Receptor Pairs'],
                                    loading_threshold=0.1,
                                    use_zscore=False,
                                    figsize=(28, 8),
                                    filename=fig_filename,
                                    row_cluster=False,
                                    tick_fontsize=12,
                                    dendrogram_ratio=0.15,
                                    
                                   )

In [ ]:
selected_factor = 'Factor 4'

In [ ]:
loading_product = c2c.analysis.tensor_downstream.get_joint_loadings(tensor.factors,
                                                                    dim1='Sender Cells',
                                                                    dim2='Receiver Cells',
                                                                    factor=selected_factor,
                                                                   )

In [ ]:
lprod_cm = c2c.plotting.loading_clustermap(loading_product.T, # Remove .T to transpose the axes
                                           use_zscore=False, # Whether standardizing the loadings across factors
                                           figsize=(8, 8),
                                           filename=output_folder + '/Clustermap-CC-Pairs.pdf',
                                           cbar_label='Loading Product',
                                          )

In [ ]:
threshold = 0.1

In [ ]:
c2c.plotting.ccc_networks_plot(tensor.factors,
                               included_factors=None,
                               ccc_threshold=threshold, # Only important communication
                               nrows=1,
                               panel_size=(12,12), # This changes the size of each figure panel.
                               node_label_size=30,
                               filename=output_folder + '/Factor-Networks.pdf',
                              )

In [ ]:
import decoupler as dc

In [ ]:
lr_loadings = tensor.factors['Ligand-Receptor Pairs']

# Generate list with ligand-receptors pairs in DB
lr_list = ['^'.join(row) for idx, row in lr_pairs.iterrows()]

# Specify the organism and pathway database to use for building the LR set
organism = "mouse"
pathwaydb = "KEGG"

# Generate ligand-receptor gene sets
lr_set = c2c.external.generate_lr_geneset(lr_list,
                                          complex_sep='_',
                                          lr_sep='^',
                                          organism=organism,
                                          pathwaydb=pathwaydb,
                                          readable_name=True,
                                          output_folder=output_folder
                                         )

In [ ]:
pvals, scores, gsea_df = c2c.external.run_gsea(loadings=lr_loadings,
                                               lr_set=lr_set,
                                               output_folder=output_folder,
                                               weight=1,
                                               min_size=15,
                                               permutations=999,
                                               processes=6,
                                               random_state=6,
                                               significance_threshold=0.05,
                                              )

In [ ]:
gsea_df.loc[(gsea_df['Adj. P-value'] <= 0.05) & (gsea_df['NES'] > 0.)]

In [ ]:
gsea_df.loc[(gsea_df['Adj. P-value'] <= 0.05) & (gsea_df['NES'] < 0.)]

In [ ]:
pathway_label = '{} Annotations'.format(pathwaydb)
fig_filename = output_folder + '/GSEA-Dotplot.pdf'

with sns.axes_style("darkgrid"):
    dotplot = c2c.plotting.pval_plot.generate_dot_plot(pval_df=pvals,
                                                      score_df=scores,
                                                      significance=0.05,
                                                      xlabel='',
                                                      ylabel='{} Annotations'.format(pathwaydb),
                                                      cbar_title='NES',
                                                      cmap='PuOr',
                                                      figsize=(8,12),
                                                      label_size=24,
                                                      title_size=24,
                                                      tick_size=20,
                                                      filename=fig_filename
                                                      )

In [ ]:
%pip install git+https://github.com/saezlab/pypath.git

In [ ]:
# We first load the PROGENy gene sets
net = dc.get_progeny(organism='mouse', top=5000)

# Then convert them to sets with weighed ligand-receptor pairs
lr_progeny = li.rs.generate_lr_geneset(lr_pairs, net, lr_sep="^")

In [ ]:
estimate, pvals =  dc.run_mlm(lr_loadings.transpose(), 
                              lr_progeny, 
                              source="source", 
                              target="interaction", 
                              use_raw=False)

In [ ]:
fig_filename = output_folder + '/PROGENy.pdf'
_ = sns.clustermap(estimate, xticklabels=estimate.columns, cmap='coolwarm', z_score=4)

t = _.ax_heatmap.set_xticklabels(_.ax_heatmap.get_xmajorticklabels(), fontsize = 16)
t = _.ax_heatmap.set_yticklabels(_.ax_heatmap.get_ymajorticklabels(), fontsize = 16, rotation=0)

plt.savefig(fig_filename, dpi=300, bbox_inches='tight')

In [ ]:
selected_factor = 'Factor 3'
fig_filename = output_folder + '/PROGENy-{}.pdf'.format(selected_factor.replace(' ', '-'))

dc.plot_barplot(estimate,
                selected_factor,
                vertical=True, 
                cmap='coolwarm', 
                save=fig_filename)